In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
import soundfile as sf
import torchaudio
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import models, transforms
from PIL import Image
import time

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# ==========================================
# 1. 통합 데이터 로더 (사진 + 주파수 + 소리 묶기)
# ==========================================
class FusionDataset(Dataset):
    def __init__(self, root_dir):
        self.samples = []
        self.target_sr = 16000
        self.max_length = 3 * 16000
        
        # 영상 이름(예: aaqaifqrwn)을 기준으로 3개 파일을 한 세트로 묶습니다.
        for label_str, label_idx in [("fake", 0), ("real", 1)]:
            audio_dir = os.path.join(root_dir, "audio_visual", label_str)
            spatial_dir = os.path.join(root_dir, "spatial", label_str)
            freq_dir = os.path.join(root_dir, "frequency", label_str)

            if not os.path.exists(audio_dir): continue

            for audio_file in os.listdir(audio_dir):
                if audio_file.endswith('.wav'):
                    video_name = audio_file.replace('.wav', '')
                    
                    # 짝꿍 파일들 경로 생성
                    img_path = os.path.join(spatial_dir, f"{video_name}_frame0000.jpg")
                    freq_path = os.path.join(freq_dir, f"{video_name}_frame0000.npy")
                    audio_path = os.path.join(audio_dir, audio_file)

                    # 3개 파일이 모두 존재하는 완벽한 세트만 학습에 사용!
                    if os.path.exists(img_path) and os.path.exists(freq_path):
                        self.samples.append({
                            'img_path': img_path, 'freq_path': freq_path, 
                            'audio_path': audio_path, 'label': label_idx
                        })

        # 이미지 및 오디오 변환기 세팅
        self.transform_img = transforms.Compose([
            transforms.Resize((224, 224)), transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ])

        self.mel_spectrogram = torchaudio.transforms.MelSpectrogram(
            sample_rate=self.target_sr,
            n_mels=128,
            n_fft=1024,
            hop_length=512
        )
        
        self.amplitude_to_db = torchaudio.transforms.AmplitudeToDB()

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        
        # 1) 사진
        img = Image.open(sample['img_path']).convert('RGB')
        spatial_tensor = self.transform_img(img)
        
        # 2) 주파수
        freq_data = np.load(sample['freq_path'])
        freq_tensor = torch.from_numpy(freq_data).float().unsqueeze(0)
        freq_tensor = (freq_tensor - freq_tensor.mean()) / (freq_tensor.std() + 1e-9)
        
        # 3) 소리 (안전한 soundfile 방식)
        audio_array, sr = sf.read(sample['audio_path'])
        waveform = torch.from_numpy(audio_array).float()
        if waveform.ndim == 1: waveform = waveform.unsqueeze(0)
        else: waveform = waveform.t()
        
        if sr != self.target_sr:
            waveform = torchaudio.functional.resample(waveform, orig_freq=sr, new_freq=self.target_sr)
        if waveform.shape[0] > 1:
            waveform = torch.mean(waveform, dim=0, keepdim=True)
            
        if waveform.shape[1] > self.max_length: waveform = waveform[:, :self.max_length]
        else: waveform = F.pad(waveform, (0, self.max_length - waveform.shape[1]))
            
        audio_tensor = self.amplitude_to_db(self.mel_spectrogram(waveform))
        audio_tensor = (audio_tensor - audio_tensor.mean()) / (audio_tensor.std() + 1e-9)

        return spatial_tensor, freq_tensor, audio_tensor, sample['label']

In [ ]:


# ==========================================
# 2. 팀장 AI (Stacking Model) 조립하기
# ==========================================
class DeepfakeStackingModel(nn.Module):
    def __init__(self):
        super(DeepfakeStackingModel, self).__init__()
        
        # A. 시각 전문가
        self.spatial = models.resnet50()
        self.spatial.fc = nn.Linear(self.spatial.fc.in_features, 2)
        self.spatial.load_state_dict(torch.load('best_spatial_model.pth', map_location=device))
        
        # B. 주파수 전문가
        self.freq = models.resnet18()
        self.freq.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.freq.fc = nn.Linear(self.freq.fc.in_features, 2)
        self.freq.load_state_dict(torch.load('best_frequency_model.pth', map_location=device))
        
        # C. 청각 전문가
        self.audio = models.resnet18()
        self.audio.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.audio.fc = nn.Linear(self.audio.fc.in_features, 2)
        self.audio.load_state_dict(torch.load('best_audio_model.pth', map_location=device))

        for expert in [self.spatial, self.freq, self.audio]:
            expert.eval()
            for param in expert.parameters():
                param.requires_grad = False

        # 3명이 각각 2개의 점수(Real/Fake)를 주므로 총 6개의 의견을 받아서 2개(최종 결론)로 줄입니다.
        # 개선: 조금 더 깊게
        self.meta_classifier = nn.Sequential(
            nn.Linear(6, 32),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(16, 2)
        )

    def forward(self, x_s, x_f, x_a):
        # 전문가들의 의견 청취
        out_s = self.spatial(x_s)
        out_f = self.freq(x_f)
        out_a = self.audio(x_a)
        
        # 의견 종합 (옆으로 이어붙임)
        combined = torch.cat((out_s, out_f, out_a), dim=1)
        
        # 팀장 최종 결재
        return self.meta_classifier(combined)

In [ ]:
root_dir = "2_Processed_Data"
fusion_dataset = FusionDataset(root_dir)

train_size = int(0.8 * len(fusion_dataset))
test_size = len(fusion_dataset) - train_size
train_dataset, test_dataset = random_split(
    fusion_dataset, 
    [train_size, test_size],
    generator=torch.Generator().manual_seed(42)  # 추가
)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

model = DeepfakeStackingModel().to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.meta_classifier.parameters(), lr=0.001)

num_epochs = 10 # 팀장은 결재하는 법만 배우면 되므로 5번이면 충분합니다.
best_acc = 0.0

print(f" 총 {len(fusion_dataset)}개의 영상으로 앙상블 팀장 AI 학습을 시작합니다!\n")
start_time = time.time()

for epoch in range(num_epochs):
    print(f'Epoch {epoch+1}/{num_epochs}')
    print('-' * 15)

    model.train()
    running_loss = 0.0; corrects = 0

    for s_in, f_in, a_in, labels in train_loader:
        s_in, f_in, a_in, labels = s_in.to(device), f_in.to(device), a_in.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(s_in, f_in, a_in)
        _, preds = torch.max(outputs, 1)
        
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * s_in.size(0)
        corrects += torch.sum(preds == labels.data)

    epoch_loss = running_loss / len(train_dataset)
    epoch_acc = corrects.double() / len(train_dataset)
    print(f'[Train] Loss: {epoch_loss:.4f} | Acc: {epoch_acc:.4f}')

    model.eval()
    test_loss = 0.0; test_corrects = 0

    with torch.no_grad():
        for s_in, f_in, a_in, labels in test_loader:
            s_in, f_in, a_in, labels = s_in.to(device), f_in.to(device), a_in.to(device), labels.to(device)
            outputs = model(s_in, f_in, a_in)
            _, preds = torch.max(outputs, 1)
            loss = criterion(outputs, labels)
            test_loss += loss.item() * s_in.size(0)
            test_corrects += torch.sum(preds == labels.data)

    test_epoch_loss = test_loss / len(test_dataset)
    test_epoch_acc = test_corrects.double() / len(test_dataset)
    print(f'[Test]  Loss: {test_epoch_loss:.4f} | Acc: {test_epoch_acc:.4f}')

    if test_epoch_acc > best_acc:
        best_acc = test_epoch_acc
        torch.save(model.state_dict(), 'best_ensemble_model.pth')
        print(" 앙상블 최고 성능 갱신! 모델 저장 완료!")
    print()

print(f' 최종 앙상블 학습 완료! 최고 테스트 정답률: {best_acc:.4f}')

 총 1302개의 영상으로 앙상블 팀장 AI 학습을 시작합니다!

Epoch 1/10
---------------
[Train] Loss: 1.3149 | Acc: 0.3468
[Test]  Loss: 0.3094 | Acc: 0.9770
 앙상블 최고 성능 갱신! 모델 저장 완료!

Epoch 2/10
---------------
[Train] Loss: 0.2502 | Acc: 0.9212
[Test]  Loss: 0.1166 | Acc: 0.9770

Epoch 3/10
---------------
[Train] Loss: 0.1374 | Acc: 0.9673
[Test]  Loss: 0.0798 | Acc: 0.9770

Epoch 4/10
---------------
[Train] Loss: 0.0929 | Acc: 0.9856
[Test]  Loss: 0.0596 | Acc: 0.9808
 앙상블 최고 성능 갱신! 모델 저장 완료!

Epoch 5/10
---------------
[Train] Loss: 0.0846 | Acc: 0.9837
[Test]  Loss: 0.0461 | Acc: 0.9847
 앙상블 최고 성능 갱신! 모델 저장 완료!

Epoch 6/10
---------------
[Train] Loss: 0.0634 | Acc: 0.9846
[Test]  Loss: 0.0385 | Acc: 0.9847

Epoch 7/10
---------------
[Train] Loss: 0.0533 | Acc: 0.9904
[Test]  Loss: 0.0322 | Acc: 0.9847

Epoch 8/10
---------------
[Train] Loss: 0.0496 | Acc: 0.9875
[Test]  Loss: 0.0273 | Acc: 0.9923
 앙상블 최고 성능 갱신! 모델 저장 완료!

Epoch 9/10
---------------
[Train] Loss: 0.0475 | Acc: 0.9866
[Test]  Loss: 0.0